# Chapter 8 — Gradient, Hessian and Taylor's Formula

Companion notebook to Chapter 8 of *Advanced Calculus for Machine Learning*.
Every numerical claim of the chapter is recomputed here with PyTorch and
checked with an assertion.

Vocabulary used below: `gamma` is the array of partial derivatives (what
`autograd` returns), `M` the Gram matrix of an inner product, `H` the Hessian
matrix. The gradient is `M^{-1} gamma`; the Hessian operator is `M^{-1} H`.

In [1]:
import math
import warnings
import torch
from torch.func import jacrev, jvp, hessian, grad

warnings.filterwarnings("ignore", category=FutureWarning)
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print(torch.__version__)

2.15.0+cu132


## 1. One covector, several gradients

Example 8.3 and Figure 8.1, then Exercise 8.1. For each inner product the
gradient satisfies `<grad, h> = df(a)(h)`, is the direction of steepest
ascent among vectors of length 1 *for that inner product*, and has length
equal to the dual norm of `df(a)`.

In [2]:
def gradient(gamma, M):
    return torch.linalg.solve(M, gamma)

gamma = torch.tensor([1.0, 2.0])
I2 = torch.eye(2); M4 = torch.diag(torch.tensor([1.0, 4.0]))
g_dot, g_M = gradient(gamma, I2), gradient(gamma, M4)
print("dot product   :", g_dot.tolist(), " length", math.sqrt(g_dot @ I2 @ g_dot))
print("M = diag(1, 4):", g_M.tolist(), " length", math.sqrt(g_M @ M4 @ g_M))
assert torch.allclose(g_M, torch.tensor([1.0, 0.5]))
assert abs((g_M @ M4 @ g_M).item() - 2.0) < 1e-12

# steepest ascent among unit vectors of each inner product (scan the unit ball)
angles = torch.linspace(0, 2 * math.pi, 200001)
dirs = torch.stack([torch.cos(angles), torch.sin(angles)], 1)
for name, M, g in (("dot", I2, g_dot), ("M", M4, g_M)):
    unit = dirs / torch.sqrt(torch.einsum("ni,ij,nj->n", dirs, M, dirs))[:, None]
    vals = unit @ gamma
    best = unit[vals.argmax()]
    gnorm = math.sqrt(g @ M @ g)
    print(f"{name}: best unit direction {[round(v, 4) for v in best.tolist()]}"
          f"  value {vals.max().item():.6f}  |grad| = {gnorm:.6f}")
    assert torch.allclose(best, g / gnorm, atol=1e-4)
    assert abs(vals.max().item() - gnorm) < 1e-8

# Exercise 8.1: f = x^2 + 3xy at (1, 1)
f = lambda p: p[0] ** 2 + 3 * p[0] * p[1]
a = torch.tensor([1.0, 1.0])
gam = jacrev(f)(a)
M3 = torch.tensor([[2.0, 1.0], [1.0, 2.0]])
g3 = gradient(gam, M3)
print("Exercise 8.1: gamma =", gam.tolist(), " gradients:",
      gradient(gam, I2).tolist(), gradient(gam, M4).tolist(), g3.tolist())
assert torch.allclose(gradient(gam, M4), torch.tensor([5.0, 0.75]))
assert torch.allclose(g3, torch.tensor([7 / 3, 1 / 3]))
assert torch.allclose(M3 @ g3, gam)              # <grad, e_i> = partial derivatives

dot product   : [1.0, 2.0]  length 2.23606797749979
M = diag(1, 4): [1.0, 0.5]  length 1.4142135623730951
dot: best unit direction [0.4472, 0.8944]  value 2.236068  |grad| = 2.236068
M: best unit direction [0.7071, 0.3535]  value 1.414214  |grad| = 1.414214


Exercise 8.1: gamma = [5.0, 3.0]  gradients: [5.0, 3.0] [5.0, 0.75] [2.3333333333333335, 0.3333333333333333]


### The metric update is the same point in every basis; the plain update is not

Equation (8.1) and Proposition 3.14. Under a change of basis `x = P x'`, the
arrays become `gamma' = P^T gamma` and `M' = P^T M P`.

In [3]:
torch.manual_seed(1)
P = torch.randn(2, 2) + 2 * torch.eye(2)
x = torch.tensor([1.0, 1.0]); eta = 0.1
xp, gp, Mp = torch.linalg.solve(P, x), P.T @ gam, P.T @ M3 @ P
metric_old = x - eta * gradient(gam, M3)
metric_new = P @ (xp - eta * gradient(gp, Mp))
plain_old = x - eta * gam
plain_new = P @ (xp - eta * gp)
print("metric update, two bases:", metric_old.tolist(), metric_new.tolist())
print("plain update,  two bases:", plain_old.tolist(), plain_new.tolist())
assert torch.allclose(metric_old, metric_new) and not torch.allclose(plain_old, plain_new)

# polar coordinates (Exercise 8.2): M = J^T J = diag(1, rho^2)
polar = lambda p: torch.stack([p[0] * torch.cos(p[1]), p[0] * torch.sin(p[1])])
pt = torch.tensor([2.0, 0.7])
J = jacrev(polar)(pt)
Mpol = J.T @ J
assert torch.allclose(Mpol, torch.diag(torch.tensor([1.0, 4.0])), atol=1e-12)
fxy = lambda q: q[0] ** 2 * q[1] + torch.sin(q[1])
g_fun = lambda p: fxy(polar(p))
comp = torch.linalg.solve(Mpol, jacrev(g_fun)(pt))       # components in (e_rho, e_theta)
assert torch.allclose(J @ comp, jacrev(fxy)(polar(pt)))  # the usual gradient in x, y
print("gradient components in polar basis:", comp.tolist(), "(second one carries 1/rho^2)")

metric update, two bases: [0.7666666666666666, 0.9666666666666667] [0.7666666666666665, 0.9666666666666667]
plain update,  two bases: [0.5, 0.7] [-2.836173228519334, -1.4944514212772397]
gradient components in polar basis: [4.70177559472828, -0.26829641347109806] (second one carries 1/rho^2)


## 2. The second differential

Proposition 8.5 (Hessian matrix), Theorem 8.6 (symmetry), Proposition 8.7
(`P^T H P` under an affine change of variable) and Exercise 8.4 (a function
whose mixed partial derivatives differ at one point).

In [4]:
f = lambda p: torch.exp(p[0]) * torch.cos(p[1]) + p[0] ** 3 * p[1] ** 2 + torch.sin(p[0] * p[2])
a = torch.tensor([0.3, -0.7, 1.1])
H = hessian(f)(a)
print("Hessian matrix:\n", H)
assert torch.allclose(H, H.T)

# D^2 f(a)(h, k) = d/dt df(a + t h)(k)
h = torch.tensor([0.5, 1.0, -0.2]); k = torch.tensor([-1.0, 0.3, 0.8])
dd = jvp(lambda p: jacrev(f)(p) @ k, (a,), (h,))[1]
assert torch.allclose(dd, h @ H @ k)

# affine change of variable x = phi(u) = o + A u
A = torch.randn(3, 3) + 2 * torch.eye(3); o = torch.tensor([0.1, 0.2, -0.3])
g = lambda u: f(o + A @ u)
u = torch.linalg.solve(A, a - o)
assert torch.allclose(jacrev(g)(u), jacrev(f)(a) @ A)       # row rule
assert torch.allclose(hessian(g)(u), A.T @ H @ A)           # bilinear-form rule
print("under x = o + A u: row -> row A,  Hessian -> A^T H A")

# Exercise 8.4: mixed partial derivatives of xy(x^2 - y^2)/(x^2 + y^2) at 0
def fx(x, y):
    return x * y * (x * x - y * y) / (x * x + y * y) if (x, y) != (0.0, 0.0) else 0.0
e = 1e-5
dx_at = lambda y: (fx(e, y) - fx(-e, y)) / (2 * e)          # d/dx f(0, y)
dy_at = lambda x: (fx(x, e) - fx(x, -e)) / (2 * e)          # d/dy f(x, 0)
dydx = (dx_at(1e-2) - dx_at(-1e-2)) / 2e-2
dxdy = (dy_at(1e-2) - dy_at(-1e-2)) / 2e-2
print("d/dy(df/dx)(0,0) =", round(dydx, 6), "  d/dx(df/dy)(0,0) =", round(dxdy, 6))
assert abs(dydx + 1) < 1e-5 and abs(dxdy - 1) < 1e-5

Hessian matrix:
 tensor([[ 1.5223,  0.4916,  0.8391],
        [ 0.4916, -0.9784,  0.0000],
        [ 0.8391,  0.0000, -0.0292]])
under x = o + A u: row -> row A,  Hessian -> A^T H A
d/dy(df/dx)(0,0) = -0.999998   d/dx(df/dy)(0,0) = 0.999998


## 3. Taylor's formula and critical points

Theorem 8.9: the remainder of the quadratic model is small compared with
`|h|^2`. Exercise 8.5 gives the numbers for `e^x cos y`. Theorem 8.10 and
Example 8.11, Exercise 8.3: critical points are classified by the signature,
which does not change under `H -> P^T H P`.

In [5]:
f = lambda p: torch.exp(p[0]) * torch.cos(p[1])
a = torch.zeros(2)
gam, H = jacrev(f)(a), hessian(f)(a)
model = lambda h: f(a) + gam @ h + 0.5 * h @ H @ h
h = torch.tensor([0.1, 0.2])
print("Exercise 8.5: model", model(h).item(), "  f", f(a + h).item(),
      "  error", (model(h) - f(a + h)).item())
assert abs(model(h).item() - 1.085) < 1e-12 and abs(f(a + h).item() - 1.0831) < 5e-5

for s in (1.0, 0.1, 0.01):
    hs = s * h
    r = abs((f(a + hs) - model(hs)).item())
    print(f"|h| = {hs.norm().item():.4f}   |r(h)| / |h|^2 = {r / hs.norm().item() ** 2:.3e}")
assert abs((f(a + 0.01 * h) - model(0.01 * h)).item()) / (0.01 * h).norm().item() ** 2 < 1e-3

def signature(H, tol=1e-9):
    lam = torch.linalg.eigvalsh(H)
    return int((lam > tol).sum()), int((lam < -tol).sum())

f11 = lambda p: p[0] ** 3 - 3 * p[0] + p[1] ** 2                 # Example 8.11
for pt, sig in ((torch.tensor([1.0, 0.0]), (2, 0)), (torch.tensor([-1.0, 0.0]), (1, 1))):
    assert torch.allclose(jacrev(f11)(pt), torch.zeros(2)) and signature(hessian(f11)(pt)) == sig
f3 = lambda p: p[0] ** 2 * p[1] - p[0] ** 2 - 2 * p[1] ** 2      # Exercise 8.3
for pt, sig in ((torch.tensor([0.0, 0.0]), (0, 2)), (torch.tensor([2.0, 1.0]), (1, 1)),
                (torch.tensor([-2.0, 1.0]), (1, 1))):
    assert torch.allclose(jacrev(f3)(pt), torch.zeros(2))
    Hc = hessian(f3)(pt)
    assert signature(Hc) == sig
    Pr = torch.randn(2, 2) + 2 * torch.eye(2)
    assert signature(Pr.T @ Hc @ Pr) == sig                      # invariant
    print("critical point", pt.tolist(), " signature", sig)

Exercise 8.5: model 1.0850000000000002   f 1.0831410796080632   error 0.0018589203919370245
|h| = 0.2236   |r(h)| / |h|^2 = 3.718e-02
|h| = 0.0224   |r(h)| / |h|^2 = 3.672e-03
|h| = 0.0022   |r(h)| / |h|^2 = 3.667e-04
critical point [0.0, 0.0]  signature (0, 2)
critical point [2.0, 1.0]  signature (1, 1)
critical point [-2.0, 1.0]  signature (1, 1)


## 4. The Hessian operator and "sharpness"

Proposition 8.13: the Hessian operator is `M^{-1} H`, the differential of the
gradient field; its eigenvalues depend on `M`. The machine-learning box that
follows: at a minimum, a reparametrization changes the eigenvalues of `H` as
much as one wishes (Exercise 8.9), while the signature stays.

In [6]:
f = lambda p: torch.cosh(p[0]) + p[0] * p[1] + 2 * p[1] ** 2
a = torch.tensor([0.4, -0.3])
H = hessian(f)(a)
M = torch.tensor([[2.0, 1.0], [1.0, 2.0]])
grad_field = lambda p: torch.linalg.solve(M, jacrev(f)(p))
assert torch.allclose(jacrev(grad_field)(a), torch.linalg.solve(M, H))    # D(grad f) = M^{-1} H
ev_dot = torch.linalg.eigvalsh(H)
ev_M = torch.linalg.eigvals(torch.linalg.solve(M, H)).real.sort().values
print("eigenvalues for the dot product:", [round(v, 4) for v in ev_dot.tolist()])
print("eigenvalues for M              :", [round(v, 4) for v in ev_M.tolist()])
assert (ev_dot > 0).all() and (ev_M > 0).all() and not torch.allclose(ev_dot, ev_M)

# Exercise 8.9: g(x) = f(c x): sharpness multiplied by c^2
f1 = lambda y: (y - 1.0) ** 2 + 0.3 * (y - 1.0) ** 4       # minimum at y = 1, f'' = 2
d2 = lambda fun, x: hessian(fun)(torch.tensor(x)).item()
for c in (0.1, 1.0, 10.0):
    print(f"c = {c:5.1f}: g'' at the minimum = {d2(lambda x: f1(c * x), 1.0 / c):.4f}")
    assert abs(d2(lambda x: f1(c * x), 1.0 / c) - 2 * c * c) < 1e-9

# a non-affine change of variable: (e^x - 1)^2 is not convex although y^2 is
g2 = lambda x: (torch.exp(x) - 1) ** 2
print("g''(0) =", d2(g2, 0.0), "  g''(-1) =", round(d2(g2, -1.0), 4))
assert abs(d2(g2, 0.0) - 2.0) < 1e-12 and d2(g2, -1.0) < 0

eigenvalues for the dot product: [0.7713, 4.3097]
eigenvalues for M              : [0.4987, 2.222]


c =   0.1: g'' at the minimum = 0.0200
c =   1.0: g'' at the minimum = 2.0000


c =  10.0: g'' at the minimum = 200.0000
g''(0) = 2.0   g''(-1) = -0.1944


## 5. Convexity, smoothness, and one step of gradient descent

Theorem 8.14 and Exercise 8.6 on log-sum-exp: its Hessian matrix is
`diag(p) - p p^T`, positive semidefinite, zero on `(1, ..., 1)`, and bounded
by `1/2` for the dot product. Then Corollary 8.18 and Theorem 8.19 on
Exercise 8.7.

In [7]:
torch.manual_seed(2)
lse = lambda z: torch.logsumexp(z, 0)
worst = 0.0
for _ in range(200):
    z = 3 * torch.randn(6)
    p = torch.softmax(z, 0)
    Hl = hessian(lse)(z)
    assert torch.allclose(Hl, torch.diag(p) - torch.outer(p, p), atol=1e-12)
    lam = torch.linalg.eigvalsh(Hl)
    assert lam[0] > -1e-12 and lam[-1] <= 0.5 + 1e-12
    assert torch.allclose(Hl @ torch.ones(6), torch.zeros(6), atol=1e-12)
    worst = max(worst, lam[-1].item())
    # above every tangent affine function
    w = 3 * torch.randn(6)
    assert lse(w) >= lse(z) + jacrev(lse)(z) @ (w - z) - 1e-12
print("largest eigenvalue seen over 200 points:", round(worst, 4), " (bound 1/2)")
z2 = torch.tensor([40.0, 40.0])
print("the bound 1/2 is approached with two equal logits:",
      torch.linalg.eigvalsh(hessian(lse)(z2))[-1].item())

largest eigenvalue seen over 200 points: 0.4951  (bound 1/2)
the bound 1/2 is approached with two equal logits: 0.49999999999999906


In [8]:
# Exercise 8.7: f = x1^2 + 4 x2^2, dot product, a0 = (2.5, 1)
f = lambda p: p[0] ** 2 + 4 * p[1] ** 2
a0 = torch.tensor([2.5, 1.0])
Lc, mu = 8.0, 2.0
g0 = jacrev(f)(a0)
a1 = a0 - g0 / Lc
print("grad =", g0.tolist(), " a1 =", a1.tolist(), " f(a0) =", f(a0).item(), " f(a1) =", f(a1).item())
assert torch.allclose(a1, torch.tensor([1.875, 0.0])) and f(a1).item() == 3.515625
assert f(a1) <= f(a0) - (g0 @ g0) / (2 * Lc)             # one-step bound: 4.6875
assert abs((f(a0) - (g0 @ g0) / (2 * Lc)).item() - 4.6875) < 1e-12
assert f(a1) <= (1 - mu / Lc) * f(a0)                     # rate: 7.6875
assert g0 @ g0 >= 2 * mu * f(a0)                          # 89 >= 41

# the decrease eta (1 - L eta / 2) |grad|^2 as a function of eta
for eta in (0.05, 0.125, 0.2, 0.25, 0.3):
    dec = (f(a0) - f(a0 - eta * g0)).item()
    bound = eta * (1 - Lc * eta / 2) * (g0 @ g0).item()
    print(f"eta = {eta:5.3f}: decrease {dec:8.4f}   guaranteed {bound:8.4f}")
    assert dec >= bound - 1e-12
# eta > 2/L: the loss can increase (here, starting on the direction of largest curvature)
b0 = torch.tensor([0.0, 1.0]); gb = jacrev(f)(b0)
print("eta = 0.3 > 2/L from (0, 1): f goes from", f(b0).item(), "to", f(b0 - 0.3 * gb).item())
assert f(b0 - 0.3 * gb) > f(b0)

grad = [5.0, 8.0]  a1 = [1.875, 0.0]  f(a0) = 10.25  f(a1) = 3.515625
eta = 0.050: decrease   3.7475   guaranteed   3.5600
eta = 0.125: decrease   6.7344   guaranteed   5.5625
eta = 0.200: decrease   6.5600   guaranteed   3.5600
eta = 0.250: decrease   4.6875   guaranteed   0.0000
eta = 0.300: decrease   1.4100   guaranteed  -5.3400
eta = 0.3 > 2/L from (0, 1): f goes from 4.0 to 7.839999999999999


### The two rates of Theorem 8.19 on random problems

A strongly convex quadratic for an inner product `M`: the constants are the
extreme eigenvalues of `M^{-1} H`, the step is `x - (1/L) M^{-1} gamma`, and
`f(a_k) - f_*` decreases at least like `(1 - mu/L)^k`. Then a non-convex
function, for which only `min |grad|^2 <= 2L (f(a_0) - f_*) / K` is
guaranteed.

In [9]:
torch.manual_seed(3)
n = 8
B = torch.randn(n, n); Hq = B @ B.T + 0.5 * torch.eye(n)
C = torch.randn(n, n); M = C @ C.T + torch.eye(n)
xstar = torch.randn(n)
fq = lambda x: 0.5 * (x - xstar) @ Hq @ (x - xstar)
ev = torch.linalg.eigvals(torch.linalg.solve(M, Hq)).real
Lc, mu = ev.max().item(), ev.min().item()
x = torch.randn(n); gaps = [fq(x).item()]
for _ in range(60):
    x = x - torch.linalg.solve(M, jacrev(fq)(x)) / Lc
    gaps.append(fq(x).item())
ratios = [gaps[k + 1] / gaps[k] for k in range(60)]
print(f"L = {Lc:.3f}  mu = {mu:.4f}  1 - mu/L = {1 - mu / Lc:.5f}  "
      f"largest observed ratio = {max(ratios):.5f}")
assert max(ratios) <= 1 - mu / Lc + 1e-12

# changing the inner product changes the condition number; M = H gives 1
ev_dot = torch.linalg.eigvalsh(Hq)
print("condition number: dot product", round((ev_dot[-1] / ev_dot[0]).item(), 1),
      "  M above", round(Lc / mu, 1), "  M = H", 1.0)

# non-convex: f(x) = sum cos(x_i) + 0.1 |x|^2, |D^2 f| <= 1.2, f >= -n
fn = lambda x: torch.cos(x).sum() + 0.1 * (x ** 2).sum()
Ln = 1.2
x = 3 * torch.randn(n); f0 = fn(x).item(); gn = []
K = 200
for _ in range(K):
    gk = jacrev(fn)(x); gn.append((gk @ gk).item()); x = x - gk / Ln
lower = -float(n)                       # a lower bound of f
print("min |grad|^2 =", f"{min(gn):.3e}", "  bound 2L(f0 - f_*)/K <=",
      f"{2 * Ln * (f0 - lower) / K:.3e}")
assert min(gn) <= 2 * Ln * (f0 - lower) / K

L = 7.895  mu = 0.0520  1 - mu/L = 0.99341  largest observed ratio = 0.96006
condition number: dot product 31.1   M above 151.7   M = H 1.0
min |grad|^2 = 9.861e-32   bound 2L(f0 - f_*)/K <= 1.494e-01


## 6. Newton's method

Proposition 8.21 and Exercise 8.8: the Newton step is the same point in every
affine coordinate system, whereas `x - eta * gamma` is not; one step solves a
quadratic problem; Newton's step is the gradient step of size 1 for the inner
product `D^2 f(a)`.

In [10]:
f = lambda p: torch.exp(p[0]) - p[0] + 2 * p[1] ** 2
a = torch.tensor([1.0, 1.0])
gam, H = jacrev(f)(a), hessian(f)(a)
newton = a - torch.linalg.solve(H, gam)
print("gamma =", gam.tolist(), " H =", H.tolist(), " next point =", newton.tolist())
assert torch.allclose(newton, torch.tensor([math.exp(-1), 0.0]))

P = torch.tensor([[1.0, 1.0], [0.0, 1.0]])
g = lambda u: f(P @ u)
u = torch.linalg.solve(P, a)
gam_u, H_u = jacrev(g)(u), hessian(g)(u)
assert torch.allclose(gam_u, torch.tensor([math.e - 1, math.e + 3]))
assert torch.allclose(H_u, torch.tensor([[math.e, math.e], [math.e, math.e + 4]]))
newton_u = u - torch.linalg.solve(H_u, gam_u)
print("in u:  next u =", newton_u.tolist(), " = point x", (P @ newton_u).tolist())
assert torch.allclose(P @ newton_u, newton)

plain_x = a - 0.1 * gam
plain_u = P @ (u - 0.1 * gam_u)
print("plain update in x:", [round(v, 3) for v in plain_x.tolist()],
      "  in u, mapped back:", [round(v, 3) for v in plain_u.tolist()])
assert torch.allclose(plain_x, torch.tensor([0.828, 0.600]), atol=5e-4)
assert torch.allclose(plain_u, torch.tensor([0.256, 0.428]), atol=5e-4)

# Newton = gradient step of size 1 for the inner product D^2 f(a)
assert torch.allclose(a - 1.0 * gradient(gam, H), newton)
# one step on the quadratic of Section 5
xq = torch.randn(n)
assert torch.allclose(xq - torch.linalg.solve(Hq, jacrev(fq)(xq)), xstar)
# a few iterations on f: the error is roughly squared at each step
x = a.clone(); errs = []
for _ in range(6):
    x = x - torch.linalg.solve(hessian(f)(x), jacrev(f)(x)); errs.append(x.norm().item())
print("distance to the minimum (0, 0):", [f"{e:.2e}" for e in errs])
assert errs[-1] < 1e-14

gamma = [1.718281828459045, 4.0]  H = [[2.718281828459045, 0.0], [0.0, 4.0]]  next point = [0.36787944117144233, 0.0]
in u:  next u = [0.36787944117144233, 0.0]  = point x [0.36787944117144233, 0.0]
plain update in x: [0.828, 0.6]   in u, mapped back: [0.256, 0.428]


distance to the minimum (0, 0): ['3.68e-01', '6.01e-02', '1.77e-03', '1.56e-06', '1.22e-12', '7.78e-17']


## 7. Hessian–vector products

`H v` without forming `H`: forward mode applied to the reverse-mode program
that computes `gamma` (`jvp` of `grad`), or reverse mode applied to the
scalar `gamma(x)^T v`. Used here for a power iteration giving the largest
eigenvalue of `H`.

In [11]:
torch.manual_seed(4)
nbig = 300
A = torch.randn(nbig, nbig) / math.sqrt(nbig); c = torch.randn(nbig)
u0 = torch.randn(nbig); u0 /= u0.norm()          # one direction of large curvature
big = lambda x: (torch.logsumexp(A @ x, 0) + 0.5 * ((x - c) ** 2).sum()
                 + 0.1 * torch.cos(x).sum() + 2.0 * (u0 @ x) ** 2)
x = torch.randn(nbig); v = torch.randn(nbig)

hvp_fwd = jvp(grad(big), (x,), (v,))[1]                       # forward over reverse
hvp_rev = grad(lambda y: grad(big)(y) @ v)(x)                 # reverse over reverse
Hfull = hessian(big)(x)
assert torch.allclose(hvp_fwd, Hfull @ v) and torch.allclose(hvp_rev, Hfull @ v)
print("H v computed in two ways, without H; both agree with the full matrix")

w = torch.randn(nbig); w /= w.norm()
for _ in range(100):
    w = jvp(grad(big), (x,), (w,))[1]; lam = w.norm().item(); w /= lam
top = torch.linalg.eigvalsh(Hfull)[-1].item()
print("power iteration:", round(lam, 6), "   largest eigenvalue of H:", round(top, 6))
print("alignment of the iterate with the planted direction:", round(abs((w @ u0).item()), 4))
assert abs(lam - top) < 1e-8 * top

H v computed in two ways, without H; both agree with the full matrix


power iteration: 4.939987    largest eigenvalue of H: 4.939987
alignment of the iterate with the planted direction: 1.0


## Summary

* `autograd` returns `gamma`; the gradient is `M^{-1} gamma` and depends on
  the inner product; the update `x - eta M^{-1} gamma` is the same point in
  every basis, `x - eta gamma` is not.
* The Hessian matrix is symmetric and changes as `P^T H P`; its signature
  classifies critical points; its eigenvalues depend on the inner product
  and on the parametrization.
* For an `L`-smooth function a gradient step of size `1/L` decreases the
  loss by at least `|grad|^2 / 2L`; under strong convexity the rate is
  `1 - mu/L`.
* Newton's step `-H^{-1} gamma` is the same point in every affine coordinate
  system.